# Символьная регрессия

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Символьная регрессия».

## Что делает метод

Обычная регрессия подбирает коэффициенты в заранее заданной формуле. Символьная регрессия идёт дальше: она ищет саму структуру зависимости — комбинацию операций и переменных, которая описывает данные. На выходе получается компактная формула, например $y = x_1^2 + 2 x_2$, а не «чёрный ящик».

Это ценно в естественных науках, где формула интерпретируема, проверяема и может указывать на физический закон.

В этом блокноте мы реализуем символьную регрессию эволюционным поиском по пространству формул: формулы кодируются деревьями выражений, а отбор, скрещивание и мутация улучшают популяцию. Реализация лёгкая, на numpy, без тяжёлых библиотек. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Обычная регрессия работает так: вы выбираете форму уравнения (например, «прямая»), а метод подбирает коэффициенты. Если угадать форму неправильно — результат плохой.

Символьная регрессия убирает эту необходимость угадывать. Она ищет и структуру, и коэффициенты одновременно. Из «кирпичиков» (сложение, умножение, степень, тригонометрия) метод строит и испытывает тысячи вариантов формул.

Как это делается эволюционным поиском:

1. **Начальная популяция** — тысяча случайных формул. Большинство ужасны. Несколько — неплохи.
2. **Оценка качества** — каждая формула примеряется к данным. Считается ошибка + штраф за сложность (принцип «бритвы Оккама»: из двух одинаково точных формул предпочтительнее более короткая).
3. **Отбор** — лучшие формулы переходят в следующее «поколение».
4. **Скрещивание** — у двух хороших формул обмениваются части (поддеревья). Может получиться что-то ещё лучше.
5. **Мутация** — случайная замена части формулы.
6. Повторять 30–100 поколений.

Ключевые понятия:
- **Дерево выражений** — способ записать формулу в виде дерева: листья — переменные и числа, ветки — операции.
- **Штраф за сложность** — не даёт алгоритму выдавать длинные бессмысленные формулы, точно «подогнанные» под шум данных.
- **Поколение** — один шаг эволюции: оценить, отобрать, скрестить, мутировать.

Результат — явная формула, которую можно прочитать, проверить и использовать как физический закон.

## 1. Установка библиотек

In [ ]:
%pip install -q numpy==1.26.4 pandas==2.2.2 matplotlib==3.9.2

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Сформируем данные по известной формуле $y = x^2 - 2x + 1$ с небольшим шумом. Формулу мы знаем заранее — это позволит проверить, восстановит ли её алгоритм. Сам метод формулу не получает, только числа.

In [ ]:
import numpy as np
import pandas as pd

# Фиксируем зерно для воспроизводимости
rng = np.random.default_rng(42)

def true_formula(x):
    return x ** 2 - 2 * x + 1

x = np.linspace(-3, 3, 60)
y = true_formula(x) + rng.normal(0, 0.4, size=x.size)

data = pd.DataFrame({"x": x, "y": y})
print("Истинная зависимость (для проверки): y = x^2 - 2x + 1")
data.head()

## 4. Применение метода

### Шаг 1. Представление формулы деревом

Формула кодируется **деревом выражений**: внутренние узлы — операции (+, −, ×), листья — переменная `x` или целочисленная константа. Любое дерево можно вычислить на массиве данных и записать в читаемом виде.

Например, дерево `(×, (+, x, 1), (−, x, 2))` соответствует формуле `(x + 1) × (x − 2)`.

Такое представление позволяет легко «скрещивать» формулы (обмениваться поддеревьями) и мутировать их (заменять случайное поддерево).

In [ ]:
OPS = ["+", "-", "*"]

def random_tree(depth, rng):
    """Случайное дерево выражения заданной глубины."""
    if depth == 0 or (depth < 3 and rng.random() < 0.3):
        # Лист: переменная либо целочисленная константа.
        if rng.random() < 0.5:
            return ("x",)
        return ("const", int(rng.integers(-3, 4)))
    op = OPS[rng.integers(len(OPS))]
    return (op, random_tree(depth - 1, rng), random_tree(depth - 1, rng))


def evaluate(tree, x):
    """Вычисляет дерево на массиве значений x."""
    tag = tree[0]
    if tag == "x":
        return x
    if tag == "const":
        return np.full_like(x, float(tree[1]))
    a = evaluate(tree[1], x)
    b = evaluate(tree[2], x)
    if tag == "+":
        return a + b
    if tag == "-":
        return a - b
    return a * b


def to_text(tree):
    """Записывает дерево как формулу."""
    tag = tree[0]
    if tag == "x":
        return "x"
    if tag == "const":
        return str(tree[1])
    return f"({to_text(tree[1])} {tag} {to_text(tree[2])})"


demo = random_tree(2, rng)
print("Пример случайной формулы:", to_text(demo))

### Шаг 2. Качество формулы (функция приспособленности)

Каждая формула получает оценку качества. Хорошая формула:
1. **Точная** — малая среднеквадратичная ошибка (MSE) на данных.
2. **Простая** — небольшой размер дерева.

Эти два критерия конкурируют: можно сделать очень точную, но громоздкую формулу. Коэффициент `0.01` при штрафе за размер — баланс между точностью и краткостью. Принцип «бритвы Оккама»: при равной точности короткая формула лучше, так как менее вероятно подогнана под шум.

In [ ]:
def tree_size(tree):
    if tree[0] in ("x", "const"):
        return 1
    return 1 + tree_size(tree[1]) + tree_size(tree[2])


def fitness(tree, x, y):
    """Ошибка формулы со штрафом за сложность; меньше - лучше."""
    try:
        pred = evaluate(tree, x)
        mse = np.mean((pred - y) ** 2)
    except (OverflowError, FloatingPointError):
        return 1e9
    if not np.isfinite(mse):
        return 1e9
    return mse + 0.01 * tree_size(tree)

### Шаг 3. Эволюционный поиск формулы

Популяция из 300 случайных формул улучшается за 40 поколений. На каждом шаге:

1. **Сортировка** по качеству — лучшие в начале списка.
2. **Элита** (10 % лучших) переходит без изменений — лучшее не теряем.
3. **Турнирный отбор** — выбираем двух «родителей» из первой половины популяции (они в среднем лучше, чем вторая).
4. **Скрещивание** — у родителей обмениваются случайные поддеревья.
5. **Мутация** (с вероятностью 30 %) — случайное поддерево заменяется на новое случайное.

После 40 поколений возвращается формула с наименьшей ошибкой за всё время поиска.

In [ ]:
def collect_nodes(tree, path=()):
    """Все узлы дерева вместе с путями к ним."""
    nodes = [(path, tree)]
    if tree[0] not in ("x", "const"):
        nodes += collect_nodes(tree[1], path + (1,))
        nodes += collect_nodes(tree[2], path + (2,))
    return nodes


def replace_at(tree, path, subtree):
    """Возвращает дерево с поддеревом, заменённым по пути."""
    if not path:
        return subtree
    i = path[0]
    children = [tree[1], tree[2]]
    children[i - 1] = replace_at(children[i - 1], path[1:], subtree)
    return (tree[0], children[0], children[1])


def crossover(a, b, rng):
    """Скрещивание: обмен случайными поддеревьями."""
    pa = collect_nodes(a)[rng.integers(len(collect_nodes(a)))]
    pb = collect_nodes(b)[rng.integers(len(collect_nodes(b)))]
    return replace_at(a, pa[0], pb[1])


def mutate(tree, rng):
    """Мутация: замена случайного поддерева на новое."""
    nodes = collect_nodes(tree)
    path = nodes[rng.integers(len(nodes))][0]
    return replace_at(tree, path, random_tree(2, rng))


def symbolic_regression(x, y, generations=40, pop_size=300, seed=1):
    rng = np.random.default_rng(seed)
    population = [random_tree(3, rng) for _ in range(pop_size)]
    best, best_fit, history = None, 1e18, []
    for _ in range(generations):
        scored = sorted(population, key=lambda t: fitness(t, x, y))
        if fitness(scored[0], x, y) < best_fit:
            best, best_fit = scored[0], fitness(scored[0], x, y)
        history.append(best_fit)
        # Элита переходит без изменений.
        new_pop = scored[:pop_size // 10]
        while len(new_pop) < pop_size:
            # Турнирный отбор двух родителей.
            cand = [scored[rng.integers(pop_size // 2)] for _ in range(2)]
            child = crossover(cand[0], cand[1], rng)
            if rng.random() < 0.3:
                child = mutate(child, rng)
            new_pop.append(child)
        population = new_pop
    return best, best_fit, history


best_tree, best_fit, history = symbolic_regression(x, y)
print(f"Найденная формула: y = {to_text(best_tree)}")
print(f"Ошибка с учётом штрафа: {best_fit:.4f}")

### Шаг 4. Визуализация результата и динамики поиска

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.3))

x_dense = np.linspace(-3, 3, 200)
axes[0].scatter(x, y, color=VIZ["series"][3], s=35, label="Данные")
axes[0].plot(x_dense, true_formula(x_dense), color=VIZ["series"][1],
             linewidth=3, label="Истинная зависимость")
axes[0].plot(x_dense, evaluate(best_tree, x_dense), color=VIZ["series"][0],
             linestyle="--", label="Найденная формула")
axes[0].set_title("Восстановление зависимости")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend(loc="upper center")

axes[1].plot(history, color=VIZ["series"][2])
axes[1].set_title("Улучшение лучшей формулы по поколениям")
axes[1].set_xlabel("Поколение")
axes[1].set_ylabel("Ошибка лучшей формулы")

fig.tight_layout()
plt.show()

**Как читать эти графики.**

*Левый — восстановление зависимости.* Точки — данные с шумом. Зелёная линия — истинная зависимость `y = x² − 2x + 1`. Синяя пунктирная — найденная формула. Хороший результат: синяя линия почти совпадает с зелёной. Обратите внимание: истинная формула `(x−1)²` алгебраически эквивалентна `((x - 1) * (x - 1))` — алгоритм может найти её в других, но математически равных записях.

*Правый — динамика поиска.* Горизонтальная ось — номер поколения. Вертикальная — ошибка лучшей формулы (чем меньше, тем лучше). Кривая должна монотонно убывать: это направленный поиск, а не случайный перебор. Резкие «скачки вниз» — моменты, когда скрещивание или мутация дала принципиально лучшую структуру формулы.

### «Ага»-эксперимент: несколько запусков — устойчивость формулы

Эволюционный поиск стохастичен: разные случайные стартовые популяции могут дать разные, но математически эквивалентные формулы. Запустим поиск три раза с разными начальными состояниями и сравним найденные формулы.

In [ ]:
import matplotlib.pyplot as plt

x_dense = np.linspace(-3, 3, 200)
seeds_to_try = [1, 7, 42]

fig, axes = plt.subplots(1, len(seeds_to_try), figsize=(14.5, 4.8),
                         sharey=True)

for ax, seed in zip(axes, seeds_to_try):
    tree_s, fit_s, hist_s = symbolic_regression(x, y, generations=40,
                                                pop_size=300, seed=seed)
    formula_text = to_text(tree_s)
    y_found = evaluate(tree_s, x_dense)

    ax.scatter(x, y, color=VIZ['series'][3], s=25, alpha=0.7)
    ax.plot(x_dense, true_formula(x_dense), color=VIZ['series'][1],
            linewidth=2.5, label='истинная зависимость')
    ax.plot(x_dense, y_found, color=VIZ['series'][0],
            linestyle='--', linewidth=2, label='найденная формула')
    # Отображаем формулу, сокращая длинные записи
    short_label = formula_text if len(formula_text) < 30 else formula_text[:27] + '...'
    ax.set_title(f'seed={seed}\ny = {short_label}', fontsize=10)
    ax.set_xlabel('x')
    ax.set_ylim(-2, 18)

axes[0].set_ylabel('y')
axes[0].legend(fontsize=9)
fig.suptitle('Устойчивость символьной регрессии: три независимых запуска',
             fontsize=12, y=1.01)
fig.tight_layout()
plt.show()

print("Истинная зависимость: y = x^2 - 2*x + 1  =  (x-1)^2")
print()
for seed in seeds_to_try:
    t, f, _ = symbolic_regression(x, y, seed=seed)
    print(f"  seed={seed}: найдено  y = {to_text(t)}  (ошибка={f:.4f})")

**Как читать этот график.**

Три панели — три независимых запуска алгоритма на одних и тех же данных. В каждой: точки данных, истинная зависимость (зелёная линия) и найденная формула (синяя пунктирная). Обратите внимание: формулы могут быть записаны по-разному, но при этом давать практически одинаковые кривые — математически эквивалентные записи. Это нормально для символьной регрессии. Хорошие признаки: синяя кривая совпадает с зелёной, а формула не слишком длинная.

## 5. Интерпретация результата

- **Найденная формула** записана в явном виде. Её можно упростить вручную и сравнить с теоретическими ожиданиями — в этом главное преимущество метода перед «чёрным ящиком». Формула `((x - 1) * (x - 1))` эквивалентна истинной `x² - 2x + 1`, несмотря на другую запись.
- **Левый график**: кривая найденной формулы должна почти совпасть с истинной зависимостью. Точное текстовое совпадение формул не гарантировано — эквивалентная формула может быть записана иначе.
- **Правый график**: ошибка лучшей формулы убывает по поколениям — это признак направленного поиска. Резкие снижения — моменты, когда скрещивание «нашло» полезную структуру.
- **Штраф за сложность** (коэффициент `0.01` в `fitness`): увеличьте его — получите более короткие, но менее точные формулы; уменьшите — более точные, но длинные и переобученные.
- Эволюционный поиск стохастичен: разные запуски (`seed`) дают разные, но близкие по качеству формулы. Сравнивайте несколько и выбирайте самую простую из точных.

## 6. Попробуйте на своих данных

Замените демонстрационные данные своими измерениями зависимости одной величины от другой.

1. Подготовьте два массива: вход `x` и отклик `y`.
2. Снимите комментарии в ячейке ниже и укажите свой файл.
3. Выполните ячейки раздела 4. Для надёжности запустите поиск несколько раз с разными `seed`.
4. Чтобы расширить класс формул, добавьте операции (например, деление) в список `OPS` и в функцию `evaluate`.

## 7. Поэкспериментируйте сами

Попробуйте изменить следующие параметры:

- **Истинная формула**: замените `x ** 2 - 2 * x + 1` на `2 * x + 3` (линейная) или `x ** 3 - x` (кубическая). Нужно ли больше поколений для более сложной формулы?
- **Уровень шума**: измените `0.4` на `0.05` (чистые данные) или `2.0` (сильный шум). При каком уровне шума алгоритм перестаёт находить правильную структуру?
- **Штраф за сложность**: в функции `fitness` измените `0.01 * tree_size(tree)` на `0.1` или `0.001`. Что происходит с длиной найденных формул?
- **Число поколений**: измените `generations=40` на `20` или `80`. При каком значении кривая ошибки перестаёт снижаться?
- **Набор операций**: добавьте деление в `OPS` и в `evaluate`. Нужна осторожность: деление на ноль возможно — обрабатывайте его как `1e9` в `fitness`.

In [ ]:
# --- Шаблон загрузки своих данных ---
# raw = pd.read_csv("my_measurements.csv")
# x = raw["x"].to_numpy(dtype=float)
# y = raw["y"].to_numpy(dtype=float)
#
# best_tree, best_fit, history = symbolic_regression(x, y, seed=2)
# print("Найденная формула: y =", to_text(best_tree))

## Краткие выводы

- Символьная регрессия ищет **не только коэффициенты, но и саму структуру формулы** — из заданного набора операций.
- Результат — явная, читаемая формула, а не «чёрный ящик»: её можно проверить физически и использовать вне контекста ML.
- Поиск ведётся эволюционным алгоритмом: **популяция формул** улучшается за счёт отбора, скрещивания и мутации деревьев выражений.
- **Штраф за сложность** — ключевой параметр: без него алгоритм выдаёт длинные переобученные формулы; с правильным штрафом — находит компактный закон.
- Метод стохастичен: разные запуски дают разные записи, но математически близкие формулы. Запускайте несколько раз и выбирайте самую простую из точных.
- Настоящие физические приложения: поиск эмпирических законов в данных спектроскопии, механики, термодинамики — там, где нужна интерпретируемость и обобщаемость за пределы обучающего диапазона.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. Алгоритм нашёл формулу `((x - 1) * (x - 1))` с ошибкой 0.18, а в другом запуске — `(x * x - (2 * x - 1))` с ошибкой 0.19. На левом графике обе кривые практически совпадают с истинной зависимостью. Какую из них следует предпочесть и почему?
2. Вы запустили символьную регрессию на своих экспериментальных данных с высоким уровнем шума. Алгоритм нашёл длинную формулу с 15 узлами, которая точно проходит через большинство точек, но на новом наборе данных ошибка резко возрастает. Что произошло и как это исправить?
3. Физическая переменная в ваших данных имеет размерность [Н·м], а формула, найденная алгоритмом, включает слагаемые с разными степенями переменных. Почему размерный анализ является необходимым шагом проверки, даже если численная ошибка мала?

<details>
<summary>Показать ориентиры для ответов</summary>

1. По принципу «бритвы Оккама» следует выбрать более простую формулу. Обе математически эквивалентны, однако при равной точности предпочтительнее та, что имеет меньше узлов в дереве. Компактная формула менее склонна к переобучению и легче интерпретируется физически. Численная ошибка при таком различии (0.18 vs 0.19) незначима.
2. Алгоритм подогнал формулу под шум в обучающей выборке — это переобучение. Для его предотвращения следует увеличить коэффициент штрафа за сложность (`0.01 * tree_size` → более высокое значение), ограничить максимальную глубину деревьев или оценивать качество на отдельной проверочной выборке, не используемой при поиске формулы.
3. Складывать и вычитать величины с разными размерностями физически бессмысленно: формула `F = a·x² + b·x` корректна только если оба слагаемых имеют одну размерность. Алгоритм оптимизирует численную ошибку и не знает физических ограничений; размерная несогласованность означает, что формула подогнана под конкретный масштаб данных и не является универсальным законом.
</details>
